In [1]:
# IMPORTS

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OrdinalEncoder, MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import joblib
import os

In [2]:
# LOADING DATA

train_df = pd.read_csv('../data/KDDdataset/KDDTrain.csv')
test_df = pd.read_csv('../data/KDDdataset/KDDTest.csv')
# attack score (difficulty_level column) removed to avoid data leakage
train_df = train_df.drop(columns=['attack', 'attack score'])
test_df = test_df.drop(columns=['attack', 'attack score'])
print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
train_df.head()

Train shape: (125973, 42)
Test shape: (22544, 42)


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,25,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,0
1,0,udp,other,SF,146,0,0,0,0,0,...,1,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,0
2,0,tcp,private,S0,0,0,0,0,0,0,...,26,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,1
3,0,tcp,http,SF,232,8153,0,0,0,0,...,255,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,0
4,0,tcp,http,SF,199,420,0,0,0,0,...,255,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0


In [3]:
# DISTRIBUTION

print("Train label distribution:\n", train_df['label'].value_counts())
print("\nTest label distribution:\n", test_df['label'].value_counts())

Train label distribution:
 label
0    67343
1    58630
Name: count, dtype: int64

Test label distribution:
 label
1    12833
0     9711
Name: count, dtype: int64


In [4]:
# FEATURES/LABELS

CATEGORICAL_FEATURES = ['protocol_type', 'service', 'flag']

X_train = train_df.drop(columns=['label'])
y_train = train_df['label']
X_test = test_df.drop(columns=['label'])
y_test = test_df['label']

NUMERIC_FEATURES = [col for col in X_train.columns if col not in CATEGORICAL_FEATURES]

In [5]:
# ENCODING

encoder = OrdinalEncoder(
    handle_unknown = 'use_encoded_value',
    unknown_value = -1
)

X_train[CATEGORICAL_FEATURES] = encoder.fit_transform(X_train[CATEGORICAL_FEATURES])
X_test[CATEGORICAL_FEATURES] = encoder.transform(X_test[CATEGORICAL_FEATURES])

In [6]:
# MIN-MAX SCALING

scaler = MinMaxScaler()

X_train[NUMERIC_FEATURES] = scaler.fit_transform(X_train[NUMERIC_FEATURES])
X_test[NUMERIC_FEATURES] = scaler.transform(X_test[NUMERIC_FEATURES])

In [7]:
# RANDOM FOREST BASELINE

# training
rf_model = RandomForestClassifier(n_estimators = 100, random_state = 67, n_jobs = -1)
rf_model.fit(X_train, y_train)
# prediction
y_pred_rf = rf_model.predict(X_test)

rf_acc = accuracy_score(y_test, y_pred_rf)
rf_f1 = f1_score(y_test, y_pred_rf)

print(f"Random Forest\tAccuracy: {rf_acc:.4f} | F1-Score: {rf_f1:.4f}")
print(classification_report(y_test, y_pred_rf, target_names = ['Normal', 'Attack']))

Random Forest	Accuracy: 0.7786 | F1-Score: 0.7649
              precision    recall  f1-score   support

      Normal       0.67      0.97      0.79      9711
      Attack       0.97      0.63      0.76     12833

    accuracy                           0.78     22544
   macro avg       0.82      0.80      0.78     22544
weighted avg       0.84      0.78      0.78     22544



In [8]:
# DEEP NEURAL NETWORK BASELINE

class DNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

In [9]:
# TRAINING DNN

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

X_train_t = torch.FloatTensor(X_train.values).to(device)
y_train_t = torch.FloatTensor(y_train.values).unsqueeze(1).to(device)
X_test_t = torch.FloatTensor(X_test.values).to(device)

loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size = 256, shuffle = True)

dnn = DNN(input_dim = X_train.shape[1]).to(device)
optimizer = torch.optim.Adam(dnn.parameters(), lr = 1e-3)
criterion = nn.BCELoss()

EPOCHS = 20
for epoch in range(EPOCHS):
    dnn.train()
    total_loss = 0
    for X_batch, y_batch in loader:
        optimizer.zero_grad()
        loss = criterion(dnn(X_batch), y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {total_loss/len(loader):.4f}")

Device: cpu


/var/folders/gc/7ldk4b_54dvch7d2y67hhrjm0000gn/T/ipykernel_3025/2065193698.py:7: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:219.)
  y_train_t = torch.FloatTensor(y_train.values).unsqueeze(1).to(device)


Epoch 5/20 - Loss: 0.0487
Epoch 10/20 - Loss: 0.0464
Epoch 15/20 - Loss: 0.0390
Epoch 20/20 - Loss: 0.0346


In [10]:
# DNN EVALUATION

dnn.eval()
with torch.no_grad():
    y_pred_dnn = (dnn(X_test_t).cpu().numpy() > 0.5).astype(int).flatten()

dnn_acc = accuracy_score(y_test, y_pred_dnn)
dnn_f1 = f1_score(y_test, y_pred_dnn)

print(f"DNN\tAccuracy: {dnn_acc:.4f} | F1-Score: {dnn_f1:.4f}")
print(classification_report(y_test, y_pred_dnn, target_names = ['Normal', 'Attack']))

DNN	Accuracy: 0.8014 | F1-Score: 0.7935
              precision    recall  f1-score   support

      Normal       0.69      0.97      0.81      9711
      Attack       0.97      0.67      0.79     12833

    accuracy                           0.80     22544
   macro avg       0.83      0.82      0.80     22544
weighted avg       0.85      0.80      0.80     22544



In [11]:
# COMPARING MODELS

results = pd.DataFrame({
    'Model': ['Random Forest', 'DNN'],
    'Accuracy': [rf_acc, dnn_acc],
    'F1-Score': [rf_f1, dnn_f1]
})
print(results.to_string(index = False))

        Model  Accuracy  F1-Score
Random Forest  0.778566  0.764883
          DNN  0.801366  0.793450


In [12]:
# SAVING MODELS AND PROCESSED DATA

os.makedirs('../models', exist_ok = True)
os.makedirs('../processed', exist_ok = True)

# Sklearn
joblib.dump(rf_model, '../models/rf_baseline.pkl')
joblib.dump(encoder, '../models/encoder.pkl')
joblib.dump(scaler, '../models/scaler.pkl')

# PyTorch
torch.save(dnn.state_dict(), '../models/dnn_baseline.pth')

# preprocessed data
X_train.to_csv('../processed/X_train.csv', index = False)
X_test.to_csv('../processed/X_test.csv', index = False)
y_train.to_csv('../processed/y_train.csv', index = False)
y_test.to_csv('../processed/y_test.csv', index = False)

print("Everything saved correctly")

Everything saved correctly
